# Data quality check

In [2]:
import sys
import duckdb

In [10]:
CHECKS = {"customer_key": """
            SELECT customer_key
            FROM wh.dim_customer 
            GROUP BY 1 HAVING COUNT(*) > 1
          """,
          "amounts are positive": """
            SELECT * FROM wh.fact_sales
            WHERE amount IS NULL OR amount <= 0
          """,
          "every sale has a known product": """
            SELECT * FROM wh.fact_sales
            WHERE product_key NOT IN (
                SELECT product_key
                FROM wh.dim_product
            )
          """,
          "every day reached the warehouse": """
            SELECT DISTINCT dt
            FROM staging.orders
            WHERE strftime(dt, '%Y%m%d')::INTEGER NOT IN (SELECT date_key FROM wh.fact_sales)
          """,
          "gold revenue matches the warehouse": """
            SELECT * FROM (SELECT SUM(revenue) AS r FROM 'data/lake/gold/daily_category_sales/*.parquet') AS g,
                (SELECT SUM(amount) AS r FROM wh.fact_sales WHERE status = 'paid') AS f
                WHERE abs(g.r - f.r) > 1            
          """
          }

In [11]:
# ,
#           "every customer has an email": """
#             SELECT * FROM wh.dim_customer WHERE email IS NULL
#           """

In [12]:
con = duckdb.connect("warehouse.duckdb", read_only=True)
failed = 0

for name, sql in CHECKS.items():
    bad = con.execute(f"SELECT count(*) FROM ({sql})").fetchone()[0]
    print(f"{'PASS' if bad == 0 else 'FAIL'} {name} ({bad} bad rows)")
    failed += bad > 0

con.close()

if failed:
    sys.exit(f"{failed} data quality check(s) failed")

print("all checks passed")

PASS customer_key (0 bad rows)
FAIL amounts are positive (981 bad rows)
PASS every sale has a known product (0 bad rows)
PASS every day reached the warehouse (0 bad rows)
PASS gold revenue matches the warehouse (0 bad rows)


SystemExit: 1 data quality check(s) failed